# ARIMA Implementation: From Theory to Production

*Part 6 of 8: Building, Diagnosing, and Forecasting*

Rachel understood ARIMA theory (Part 5). Now: implementation!

![ARIMA Workflow](arima_implementation_workflow.png)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 8)
np.random.seed(42)

## Step 1: Determine d (Differencing Order)

Test stationarity with ADF and KPSS.

In [ ]:
# Generate data
n = 500
drift = 0.0005
innovations = np.random.normal(0, 0.02, n)
log_returns = drift + innovations
prices = 100 * np.exp(np.cumsum(log_returns))

dates = pd.date_range('2020-01-01', periods=n, freq='D')
df = pd.DataFrame({'price': prices, 'returns': log_returns}, index=dates)

print(f"Data shape: {df.shape}")
df.head()

In [ ]:
def test_stationarity(series, name='Series'):
    """Test stationarity with ADF and KPSS"""
    print(f"\n{'='*60}")
    print(f"Testing: {name}")
    print('='*60)
    
    # ADF test
    adf_result = adfuller(series.dropna())
    print(f"\nADF Test:")
    print(f"  p-value: {adf_result[1]:.6f}")
    print(f"  {'✓ STATIONARY' if adf_result[1] < 0.05 else '✗ NON-STATIONARY'}")
    
    # KPSS test
    kpss_result = kpss(series.dropna(), regression='c')
    print(f"\nKPSS Test:")
    print(f"  p-value: {kpss_result[1]:.6f}")
    print(f"  {'✓ STATIONARY' if kpss_result[1] >= 0.05 else '✗ NON-STATIONARY'}")
    
    return adf_result[1] < 0.05, kpss_result[1] >= 0.05

# Test prices (should be non-stationary)
test_stationarity(df['price'], 'Stock Prices')

# Test returns (should be stationary)
test_stationarity(df['returns'], 'Returns')

## Step 2: Identify p and q (ACF/PACF)

![ACF PACF](acf_pacf_analysis.png)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(df['returns'].dropna(), lags=40, ax=axes[0])
axes[0].set_title('ACF: Returns', fontsize=13, fontweight='bold')

plot_pacf(df['returns'].dropna(), lags=40, ax=axes[1], method='ywm')
axes[1].set_title('PACF: Returns', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nLook for patterns:")
print("• ACF cuts off at lag q → MA(q)")
print("• PACF cuts off at lag p → AR(p)")
print("• Both decay → ARMA(p,q)")

## Step 3: Grid Search

![Grid Search](grid_search_visualization.png)

Try multiple (p,d,q) combinations, compare AIC.

In [ ]:
def grid_search_arima(series, p_values, d_values, q_values):
    """Grid search for best ARIMA parameters"""
    print("\nARIMA Grid Search...")
    
    results = []
    
    for p in p_values:
        for d in d_values:
            for q in q_values:
                try:
                    model = ARIMA(series, order=(p, d, q))
                    model_fit = model.fit()
                    
                    results.append({
                        'order': (p, d, q),
                        'AIC': model_fit.aic,
                        'BIC': model_fit.bic
                    })
                    print(f"  ARIMA{(p,d,q)}: AIC={model_fit.aic:.2f} ✓")
                except:
                    print(f"  ARIMA{(p,d,q)}: Failed ✗")
    
    results_df = pd.DataFrame(results).sort_values('AIC')
    
    print(f"\n🏆 Best model: ARIMA{results_df.iloc[0]['order']}")
    print(f"   AIC: {results_df.iloc[0]['AIC']:.2f}")
    
    return results_df.iloc[0]['order'], results_df

# Search
best_order, results = grid_search_arima(
    df['returns'],
    p_values=range(0, 3),
    d_values=[0],
    q_values=range(0, 3)
)

## Step 4: Fit Best Model

In [ ]:
# Fit best model
model = ARIMA(df['returns'], order=best_order)
model_fit = model.fit()

print("\n✅ Model Summary:")
print(model_fit.summary())

## Step 5: Diagnostics

![Diagnostics](diagnostic_tests_guide.png)

Check residuals for:
1. Mean ≈ 0
2. No autocorrelation (Ljung-Box)
3. Normality
4. Constant variance

In [ ]:
residuals = model_fit.resid

# Diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Residuals over time
axes[0, 0].plot(residuals, linewidth=1, alpha=0.8)
axes[0, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Residuals Over Time')

# Histogram
axes[0, 1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Residual Distribution')

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')

# ACF of residuals
plot_acf(residuals, lags=30, ax=axes[1, 1])
axes[1, 1].set_title('ACF of Residuals')

plt.tight_layout()
plt.show()

# Statistical tests
print("\n📊 Diagnostic Tests:")
print(f"1. Mean: {residuals.mean():.6f} (should be ~0)")

lb_test = acorr_ljungbox(residuals, lags=10, return_df=True)
print(f"2. Ljung-Box p-value: {lb_test['lb_pvalue'].iloc[-1]:.6f}")
print(f"   {'✓ No autocorrelation' if lb_test['lb_pvalue'].iloc[-1] > 0.05 else '✗ Autocorrelation present'}")

jb_stat, jb_pvalue = stats.jarque_bera(residuals)
print(f"3. Jarque-Bera p-value: {jb_pvalue:.6f}")
print(f"   {'✓ Normal' if jb_pvalue > 0.05 else '⚠ Not normal (OK if large sample)'}")

## Step 6: Forecast

In [ ]:
# Forecast next 30 periods
forecast_result = model_fit.get_forecast(steps=30)
forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int()

# Plot
plt.figure(figsize=(16, 6))

# Historical
plt.plot(df.index[-100:], df['returns'].iloc[-100:], 
         label='Historical', linewidth=2)

# Forecast
forecast_index = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1),
                               periods=30, freq='D')
plt.plot(forecast_index, forecast_mean, 
         label='Forecast', linewidth=2, linestyle='--', color='red')

# Confidence interval
plt.fill_between(forecast_index, 
                 forecast_ci.iloc[:, 0], 
                 forecast_ci.iloc[:, 1],
                 alpha=0.2, color='red')

plt.axvline(x=df.index[-1], color='black', linestyle=':', alpha=0.5)
plt.title('ARIMA Forecast with 95% Confidence Interval', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Returns')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Forecast complete!")

## SARIMA Example

![SARIMA Components](sarima_components_explained.png)

For seasonal data: SARIMA(p,d,q)(P,D,Q,s)

In [ ]:
# Generate seasonal data
n_seasonal = 365 * 2
time = np.arange(n_seasonal)
trend = 0.05 * time
seasonal = 10 * np.sin(2 * np.pi * time / 365)
noise = np.random.normal(0, 2, n_seasonal)
seasonal_data = 100 + trend + seasonal + noise

dates_seasonal = pd.date_range('2021-01-01', periods=n_seasonal, freq='D')
df_seasonal = pd.DataFrame({'value': seasonal_data}, index=dates_seasonal)

# Fit SARIMA (using smaller period for demo)
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(df_seasonal['value'],
                       order=(1, 1, 1),
                       seasonal_order=(1, 1, 1, 12))
sarima_fit = sarima_model.fit(disp=False)

print("\n✅ SARIMA fitted!")
print(f"AIC: {sarima_fit.aic:.2f}")

## Key Takeaways

✅ **7-Step Workflow**: Determine d → Identify p,q → Grid search → Fit → Diagnose → Forecast → Evaluate

✅ **Diagnostics are critical**: Always check residuals

✅ **Grid search**: Don't guess parameters

✅ **SARIMA**: Extends to seasonal data

✅ **Production**: Auto-refit, monitor errors

## What's Next

**Part 7**: Advanced Topics (GARCH, VAR, Prophet)

Now you can build production ARIMA models!